# Run 1: VGG16 + GeM / robust augmentation

Notebook độc lập cho **Colab GPU**, không cần clone repo hoặc cài requirements cũ.

**Cách chạy:** chọn GPU → điền `SFM_ZIP` trong cell cấu hình → Run all → cấp quyền mount Drive.
ZIP chứa SfM subset seed42 khoảng 16.692 ảnh và metadata. Giải nén vào `/content`, không đọc JPEG từ Drive trong training.
Log, checkpoint và lịch sử lưu trực tiếp về Drive trong `OUTPUT_ROOT/RUN_ID`.

- Run 1: VGG16–GeM, augmentation chỉ ở training.
- Run 2: cùng cấu hình run 1, thêm spatial attention trước GeM.
- Hai run bắt đầu độc lập từ cùng Caffe pretrained weights; **không resume run 2 từ run 1**.
- Mục tiêu 50 epoch tổng cộng (tính cả epoch trong checkpoint), 2.000 tuples/epoch, 400 Adam updates/epoch, BF16 (GeM/loss FP32), cạnh dài ≤1024, 5 negatives.
- Tốc độ và VRAM chưa được xác minh trên Colab GPU. Smoke ở 1024 chạy trước training; không tự đổi precision/resolution khi OOM.
- Resume chỉ ở ranh giới epoch: epoch đang chạy khi runtime mất sẽ phải chạy lại.

Bạn chỉ cần ZIP SfM để train. Oxford/Paris ZIP là tùy chọn; không có benchmark thì notebook không công bố mAP.


## 1. Cấu hình — chỉ sửa cell này trước khi Run all

`SFM_ZIP`: đường dẫn ZIP thật trên Drive, ví dụ `/content/drive/MyDrive/datasets/sfm100.zip`.
ZIP phải có `retrieval-SfM-120k.pkl` và `ims/` cùng thư mục (cho phép một thư mục cha bên ngoài).
Trong `ims/` có thể dùng cây CID chuẩn hoặc ảnh đặt phẳng. Không tự tạo lại subset từ full SfM.

Weights: nếu `WEIGHTS_PATH=None`, tải đúng Caffe VGG16 feature weights từ nguồn gốc, kiểm tra SHA256 và cache trên Drive.
Nếu bạn đã có file `.pth`, điền đường dẫn vào `WEIGHTS_PATH`.

`CU_PER_HOUR`: nhập Usage rate thật trong Colab để notebook ước lượng giới hạn 110 CU cho run này.
Để `None` thì chỉ có giới hạn giờ/session; **không thể tự bảo đảm ngân sách 300 CU**.
`SESSION_HOURS=12` là giới hạn mềm cho phiên notebook; không thay đổi giới hạn do Colab áp dụng. Notebook dừng trước epoch tiếp theo nếu dự tính không đủ giờ.
Run này đặt `EPOCHS=50` tổng cộng, không phải train thêm 50 epoch.

**Train tiếp:** giữ đúng RUN_ID của thư mục checkpoint trên Drive, `RESUME="auto"`, chỉ tăng EPOCHS và thời lượng operational. Không đổi RUN_ID sang tên có "100ep" nếu muốn tiếp tục run cũ. Giữ source/model/augmentation/precision/LR/microbatch và dataset như checkpoint; không bỏ kiểm tra identity.
Nếu checkpoint hoàn tất epoch 20 và target là 100, controller khôi phục Adam/RNG/LR rồi bắt đầu epoch 21, chạy thêm 80 epoch. Giữ toàn bộ thư mục run, gồm latest.pt, hash và các best_epoch_*.pt.


In [ ]:
# === DRIVE INPUTS ===
SFM_ZIP = "/content/drive/MyDrive/IIR_dataset/prepared_subset100.zip"
BENCHMARK_ZIPS = ["/content/drive/MyDrive/IIR_dataset/iir-b0-benchmarks.zip"]  # One ZIP containing both Oxford5k and Paris6k
WEIGHTS_PATH = None  # Optional: exact Drive .pth path; None = download + verified Drive cache

OUTPUT_ROOT = "/content/drive/MyDrive/SEG301/colab_runs"
RUN_ID = "run1_vgg16_gem_augmentation"  # Keep this unchanged to resume; new name = independent fresh run
RESUME = "auto"  # None = fresh; after interruption keep RUN_ID and switch to "auto"

# Optional exact paths inside the extracted /content tree; normally leave None.
SFM_METADATA_OVERRIDE = None
SFM_IMAGES_OVERRIDE = None

# === SHARED EXPERIMENT CONFIG ===
EPOCHS = 50
SEED = 42
IMAGE_SIZE = 1024
QUERY_SIZE = 2000
POOL_SIZE = 20000  # Clipped to actual 15,953 training images
BATCH_SIZE = 5  # tuples; q+p+5 negatives per tuple; summed gradients
LR = 1e-6
WEIGHT_DECAY = 1e-6
MARGIN = 0.7
WORKERS = 4
MICROBATCH = 8  # same-shape images only; measured stable baseline on A100
MAX_BATCH_PIXELS = 8 * 1024 * 1024
EXTRACTION_WINDOW = 8
PREFETCH_FACTOR = 2
TUPLE_FUSION = 1  # 1 / 2 / 5 tuples per backward; Adam still steps after five tuples
CACHE_MODE = "off"  # off / bytes (compressed JPEG) / decoded (resized RGB uint8, before augmentation)
CACHE_GIB = 0.0  # Read-only parent RAM cache cap; apply profiler recommendation for a NEW run
PRECISION = "BF16"  # Confirmed A100 baseline; GeM/loss, weights and Adam stay FP32
ATTENTION = False  # Run 1=False, Run 2=True
ATTENTION_LR = 1e-4  # separate small newly initialized spatial gate; backbone LR stays 1e-6

# === OPERATIONAL BUDGET / EVALUATION ===
CU_PER_HOUR = 5.4  # User-reported Colab Usage rate; update if a future runtime differs
RUN_CU_BUDGET = 110.0
SESSION_HOURS = 12
EVAL_EVERY = 3  # Oxford/Paris only if supplied; clean validation still runs every epoch
FINAL_EVALUATION = True  # best.pt, plus saved descriptors for later CPU search experiments
RUN_TRAINING = True


## 2. Mount Drive

Giữ torch sẵn có của Colab; không cài torch==1.0.0 hoặc torchvision==0.2.1.
Runtime dựng đúng VGG16 convolutional stack, bỏ classifier và max-pooling cuối; không phụ thuộc torchvision.
Code được trình bày trực tiếp trong các cell bên dưới, không nhúng thành chuỗi escape.
Các cell `%%writefile` chỉ ghi code vào module local; cell import sau đó mới nạp module để chạy.
Chạy tuần tự/Run all; nếu sửa một cell code, chạy lại toàn bộ nhóm từ cell đầu để tránh append lặp.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import sys
from google.colab import drive
drive.mount('/content/drive')

assert SFM_ZIP, "Điền SFM_ZIP ở cell cấu hình trước khi chạy."
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID), "RUN_ID chỉ gồm chữ, số, _ và -."
assert Path(OUTPUT_ROOT).resolve().is_relative_to(Path('/content/drive').resolve()), "Output phải nằm trên Drive."
assert ATTENTION is False, "Notebook này dành riêng cho run 1."
assert EPOCHS > 0 and BATCH_SIZE == 5 and QUERY_SIZE % BATCH_SIZE == 0
assert IMAGE_SIZE == 1024, "Đổi resolution là thí nghiệm khác; sửa rõ protocol nếu chủ động thay."
assert WORKERS >= 0 and MICROBATCH >= 1 and EVAL_EVERY >= 1
assert MAX_BATCH_PIXELS >= IMAGE_SIZE**2 and EXTRACTION_WINDOW >= 1
assert PREFETCH_FACTOR >= 1 and CACHE_MODE in ["off", "bytes", "decoded"]
assert TUPLE_FUSION in [1, 2, 5]
assert CACHE_MODE == "off" or CACHE_GIB > 0
assert PRECISION in ["FP32", "TF32", "BF16"]
assert CU_PER_HOUR is None or CU_PER_HOUR > 0
assert RUN_CU_BUDGET > 0 and SESSION_HOURS > 0

RUN_DIR = Path(OUTPUT_ROOT) / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CODE = Path('/content/iir_colab_code')
LOCAL_CODE.mkdir(exist_ok=True)


### 2.1. Tiện ích, log Drive và seed

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile /content/iir_colab_code/colab_runtime.py
"""Self-contained Colab retrieval training; embedded by build_notebooks.py.

Single-device FP32 VGG16-GeM, clean mining, train-only augmentation, Drive
epoch-boundary checkpoints. No Kaggle credentials or repository install needed.
"""
import copy
import gc
import hashlib
import io
import json
import math
import os
import pickle
import random
import shutil
import stat
import sys
import time
import traceback
import zipfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageFilter
from torch import nn
from torch.utils.data import Dataset, DataLoader

VERSION = 'colab_b0_robust_v1'
WEIGHTS_HASH = 'd369c8e593250e421eda192512263dccd35edb5c4b57e97312651bac10a51b7d'
WEIGHTS_URL = 'https://cmp.felk.cvut.cz/cnnimageretrieval/data/networks/imagenet/imagenet-caffe-vgg16-features-d369c8e.pth'
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
IMAGE_CACHE = {}
IMAGE_CACHE_INFO = {}
IMAGE_CACHE_KEY = None


def system_memory():
    """Effective physical/cgroup memory; cgroup headroom is conservative."""
    import psutil
    memory = psutil.virtual_memory()
    total, available = memory.total, memory.available
    for limit_path, usage_path in [('/sys/fs/cgroup/memory.max','/sys/fs/cgroup/memory.current'),
                                    ('/sys/fs/cgroup/memory/memory.limit_in_bytes','/sys/fs/cgroup/memory/memory.usage_in_bytes')]:
        try:
            limit = int(Path(limit_path).read_text().strip())
            usage = int(Path(usage_path).read_text().strip())
            if limit < total:
                total, available = limit, min(available,max(0,limit-usage))
                break
        except (OSError,ValueError):
            continue
    return dict(total_bytes=total,available_bytes=available,used_bytes=total-available)


def loader_options(cfg):
    workers = cfg['workers']
    options = dict(num_workers=workers)
    if workers:
        options['prefetch_factor'] = cfg.get('prefetch_factor',2)
        # Cache is read-only parent memory, shared by fork (not pickled per worker).
        if cfg.get('cache_mode','off') != 'off':
            options['multiprocessing_context'] = 'fork'
    return options


def prepare_image_cache(paths,cfg):
    """Bounded read-only compressed/RGB cache. No augmented image/tensor is cached."""
    global IMAGE_CACHE, IMAGE_CACHE_INFO, IMAGE_CACHE_KEY
    mode = cfg.get('cache_mode','off')
    if mode not in {'off','bytes','decoded'}:
        raise ValueError('cache_mode must be off, bytes or decoded')
    unique = list(dict.fromkeys(map(str,paths)))
    key = (mode,cfg['image_size'],cfg.get('cache_gib',0),hash(tuple(unique)))
    if IMAGE_CACHE_KEY == key:
        return dict(IMAGE_CACHE_INFO,reused=True)
    IMAGE_CACHE = {}
    IMAGE_CACHE_INFO = {}
    IMAGE_CACHE_KEY = None
    gc.collect()
    started = time.monotonic()
    memory = system_memory()
    reserve = max(4*1024**3,int(memory['total_bytes']*.2))
    budget = max(0,min(int(cfg.get('cache_gib',0)*1024**3),memory['available_bytes']-reserve))
    if mode != 'off' and budget <= 0:
        raise MemoryError('Insufficient host RAM for configured cache and 20%/4 GiB reserve')
    stored = 0

    def read(path):
        if mode == 'bytes':
            return path,Path(path).read_bytes()
        with Image.open(path) as source:
            image = source.convert('RGB')
            image.thumbnail((cfg['image_size'],cfg['image_size']),Image.Resampling.LANCZOS)
            value = np.array(image,dtype=np.uint8,copy=True)
            value.flags.writeable = False
        return path,value

    if mode != 'off':
        print(f'Building {mode} cache, cap {budget/1024**3:.1f} GiB; JPEG/RGB before augmentation.',flush=True)
        with ThreadPoolExecutor(max_workers=min(8,os.cpu_count() or 1)) as executor:
            full = False
            for start in range(0,len(unique),16):
                if system_memory()['available_bytes'] < reserve:
                    break
                for path,value in executor.map(read,unique[start:start+16]):
                    size = len(value) if mode == 'bytes' else value.nbytes
                    if stored+size > budget:
                        full = True
                        continue
                    IMAGE_CACHE[path] = value
                    stored += size
                if start and start % 1024 == 0:
                    print(f'Cache: {len(IMAGE_CACHE)}/{len(unique)} images, {stored/1024**3:.1f} GiB',flush=True)
                if full:
                    break
    IMAGE_CACHE_KEY = key
    IMAGE_CACHE_INFO = dict(mode=mode,image_size=cfg['image_size'],stored_bytes=stored,
        cached_images=len(IMAGE_CACHE),total_images=len(unique),
        coverage=len(IMAGE_CACHE)/max(1,len(unique)),budget_bytes=budget,
        build_seconds=time.monotonic()-started,reused=False)
    print('Host image cache:',IMAGE_CACHE_INFO,flush=True)
    return dict(IMAGE_CACHE_INFO)


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    with temp.open('w') as f:
        json.dump(value, f, indent=2, ensure_ascii=False, allow_nan=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(temp, path)


class Tee:
    def __init__(self, original, logfile):
        self.original, self.logfile = original, logfile

    def write(self, text):
        self.original.write(text)
        self.logfile.write(text)
        if '\n' in text:
            self.logfile.flush()
        return len(text)

    def flush(self):
        self.original.flush()
        self.logfile.flush()

    def __getattr__(self, name):
        return getattr(self.original, name)


def start_logging(run_dir):
    """Append directly to Drive, including notebook tracebacks and stderr."""
    stop = getattr(sys, '_iir_stop_logging', None)
    if stop:
        stop()
    log_dir = Path(run_dir) / 'logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    path = log_dir / f'session_{time.time_ns()}.log'
    stream = path.open('a', buffering=1)
    stdout, stderr = sys.stdout, sys.stderr
    sys.stdout, sys.stderr = Tee(stdout, stream), Tee(stderr, stream)

    def close():
        sys.stdout, sys.stderr = stdout, stderr
        stream.flush()
        stream.close()
        sys._iir_stop_logging = None

    sys._iir_stop_logging = close
    print('Drive session log:', path, flush=True)
    return path


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32))
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)




### 2.2. Đọc ZIP và kiểm tra SfM / weights

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
def extract_zip(source, cache_root):
    """Copy ZIP to local disk, validate paths/CRC, publish only complete extraction."""
    source, cache_root = Path(source), Path(cache_root)
    if not source.is_file() or source.suffix.lower() != '.zip':
        raise FileNotFoundError(f'Expected ZIP on Drive: {source}')
    info = source.stat()
    identity = dict(path=str(source.resolve()), bytes=info.st_size, mtime_ns=info.st_mtime_ns)
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]
    destination = cache_root / key
    marker = destination / '.extraction_complete.json'
    if marker.exists() and json.loads(marker.read_text())['source'] == identity:
        print('Reuse completed local extraction:', destination)
        return destination
    cache_root.mkdir(parents=True, exist_ok=True)
    local_zip = cache_root / f'{key}.zip'
    partial = cache_root / f'{key}.partial'
    if partial.exists():
        shutil.rmtree(partial)
    if destination.exists():
        raise RuntimeError(f'Incomplete extraction at {destination}; remove that local folder before retrying.')
    try:
        with zipfile.ZipFile(source) as archive:
            entries = archive.infolist()
            needed = sum(e.file_size for e in entries) + info.st_size + 2 * 1024**3
            if shutil.disk_usage(cache_root).free < needed:
                raise OSError(f'ZIP needs approximately {needed / 1024**3:.1f} GiB free local disk.')
        print('Copying ZIP Drive -> local:', source, flush=True)
        shutil.copyfile(source, local_zip)
        digest = sha256(local_zip)
        partial.mkdir()
        with zipfile.ZipFile(local_zip) as archive:
            seen = set()
            for entry in archive.infolist():
                target = (partial / entry.filename).resolve()
                if (not target.is_relative_to(partial.resolve()) or '\\' in entry.filename
                        or stat.S_ISLNK(entry.external_attr >> 16)):
                    raise ValueError(f'Unsafe ZIP entry: {entry.filename}')
                if entry.filename in seen:
                    raise ValueError(f'Duplicate ZIP entry: {entry.filename}')
                seen.add(entry.filename)
                archive.extract(entry, partial)  # zipfile validates CRC while reading.
        atomic_json(partial / marker.name, dict(source=identity, zip_sha256=digest))
        partial.rename(destination)
        print('Extraction complete:', destination, flush=True)
        return destination
    finally:
        local_zip.unlink(missing_ok=True)


def only_match(root, pattern, override=None):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = list(Path(root).rglob(pattern))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {pattern}, found {len(matches)}. Set an exact override. {matches[:8]}')
    return matches[0]


def cid_path(cid, root):
    return Path(root) / cid[-2:] / cid[-4:-2] / cid[-6:-4] / cid


def load_sfm(root, metadata_override=None, images_override=None):
    metadata = only_match(root, 'retrieval-SfM-120k.pkl', metadata_override)
    with metadata.open('rb') as f:
        database = pickle.load(f)  # Only load your trusted dataset archive.
    expected = dict(train=(15953, 100, 34337), val=(739, 20, 181))
    for split, counts in expected.items():
        part = database[split]
        actual = (len(part['cids']), len(set(part['cluster'])), len(part['qidxs']))
        if actual != counts:
            raise ValueError(f'{split} expected prepared subset counts {counts}, got {actual}. '
                             'Use the prepared seed42 subset, not full/whitening metadata.')
        assert len(part['cluster']) == len(part['cids'])
        assert len(part['qidxs']) == len(part['pidxs'])
        assert len(set(part['cids'])) == len(part['cids'])
        for cid in part['cids']:
            assert isinstance(cid, str) and Path(cid).name == cid and '\\' not in cid
        for q, p in zip(part['qidxs'], part['pidxs']):
            assert 0 <= q < len(part['cids']) and 0 <= p < len(part['cids'])
            assert q != p and part['cluster'][q] == part['cluster'][p]
    assert not set(database['train']['cids']) & set(database['val']['cids'])
    assert not set(database['train']['cluster']) & set(database['val']['cluster'])
    images = Path(images_override) if images_override else metadata.parent / 'ims'
    # Accept canonical CID folders or a flat ims directory; reject mixed layouts.
    all_cids = database['train']['cids'] + database['val']['cids']
    canonical = [cid_path(cid, images) for cid in all_cids]
    flat = [images / cid for cid in all_cids]
    if all(p.is_file() for p in canonical):
        paths = canonical
    elif all(p.is_file() for p in flat):
        paths = flat
    else:
        missing = [str(p) for p in canonical if not p.is_file()]
        raise FileNotFoundError(f'SfM ZIP incomplete or wrong ims path; missing {len(missing)}. {missing[:5]}')
    manifest = metadata.parent / 'prepared_manifest.json'
    if manifest.exists():
        prepared = json.loads(manifest.read_text())
        assert prepared['format'] == 'b0_prepared_subset_v1'
        for name, expected_hash in prepared['metadata_files'].items():
            assert Path(name).name == name
            assert sha256(metadata.parent / name) == expected_hash, name
        for relative in random.Random(42).sample(sorted(prepared['sfm_images']), 64):
            actual = (images / relative).resolve()
            assert actual.is_relative_to(images.resolve())
            assert sha256(actual) == prepared['sfm_images'][relative]['sha256'], relative
    # All paths checked, sample decode. Unsampled corruption remains a possible loader error.
    for path in random.Random(42).sample(paths, min(128, len(paths))):
        with Image.open(path) as im:
            im.convert('RGB').load()
    offset = len(database['train']['cids'])
    partitions = {}
    for split, split_paths in [('train', paths[:offset]), ('val', paths[offset:])]:
        partitions[split] = dict(database[split], paths=[str(p) for p in split_paths])
    print('SfM verified: train=15953 images/34337 pairs, val=739 images/181 pairs.')
    return partitions, dict(metadata_sha256=sha256(metadata),
                            manifest_sha256=sha256(manifest) if manifest.exists() else None)


def resolve_weights(path, cache_root):
    if path:
        weights = Path(path)
    else:
        weights = Path(cache_root) / 'imagenet-caffe-vgg16-features-d369c8e.pth'
        weights.parent.mkdir(parents=True, exist_ok=True)
        if not weights.exists():
            print('Downloading original Caffe VGG16 features:', WEIGHTS_URL)
            temp = weights.with_suffix('.download')
            torch.hub.download_url_to_file(WEIGHTS_URL, str(temp))
            if sha256(temp) != WEIGHTS_HASH:
                temp.unlink()
                raise ValueError('Downloaded weights hash mismatch')
            temp.replace(weights)
    if not weights.is_file() or sha256(weights) != WEIGHTS_HASH:
        raise ValueError(f'Expected original Caffe VGG16 feature weights SHA256 {WEIGHTS_HASH}: {weights}')
    return weights




### 2.3. Model VGG16, GeM và spatial attention

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
class SpatialAttention(nn.Module):
    """CBAM-style spatial-only gate, identity at initialization (2*sigmoid)."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3)
        nn.init.zeros_(self.conv.weight)
        nn.init.zeros_(self.conv.bias)

    def forward(self, x):
        summary = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], dim=1)
        return x * (2 * self.conv(summary).sigmoid())


class RetrievalModel(nn.Module):
    def __init__(self, attention=False, features=None, output_dim=512):
        super().__init__()
        if features is None:
            layers, channels = [], 3
            # Exactly torchvision VGG16 features[:-1], without allocating the classifier.
            for item in [64, 64, 'M', 128, 128, 'M', 256, 256, 256, 'M',
                         512, 512, 512, 'M', 512, 512, 512]:
                if item == 'M':
                    layers.append(nn.MaxPool2d(2, 2))
                else:
                    layers.extend([nn.Conv2d(channels, item, 3, padding=1), nn.ReLU(inplace=True)])
                    channels = item
            features = nn.Sequential(*layers)
        self.features = features
        self.attention = SpatialAttention() if attention else nn.Identity()
        self.p = nn.Parameter(torch.tensor([3.0]))
        self.output_dim = output_dim
        self.precision = 'fp32'

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, dtype=torch.bfloat16,
                            enabled=self.precision == 'bf16'):
            x = self.attention(self.features(x))
        # GeM powers, normalization and contrastive loss stay FP32 under BF16.
        x = x.float()
        x = x.clamp(min=1e-6).pow(self.p).mean((-2, -1)).pow(1 / self.p)
        x = x / (x.norm(dim=1, keepdim=True) + 1e-6)
        return x.T


def make_model(cfg, weights, device):
    configure_acceleration(cfg)
    model = RetrievalModel(attention=cfg['attention'])
    model.features.load_state_dict(torch.load(weights, map_location='cpu', weights_only=True), strict=True)
    model.precision = cfg.get('precision', 'FP32').lower()
    if model.precision == 'tf32':
        model.precision = 'fp32'
    return model.to(device)


def configure_acceleration(cfg):
    precision = cfg.get('precision', 'FP32').upper()
    if precision not in {'FP32', 'TF32', 'BF16'}:
        raise ValueError('precision must be FP32, TF32 or BF16')
    if precision == 'BF16' and torch.cuda.is_available() and not torch.cuda.is_bf16_supported():
        raise RuntimeError('This CUDA device does not support BF16.')
    torch.backends.cuda.matmul.allow_tf32 = precision == 'TF32'
    torch.backends.cudnn.allow_tf32 = precision == 'TF32'




### 2.4. Augmentation và đọc ảnh

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
class ImageTransform:
    def __init__(self, augment=False):
        self.augment = augment

    def __call__(self, image):
        if self.augment:
            if random.random() < 0.8:
                operations = [(ImageEnhance.Brightness, 0.8, 1.2),
                              (ImageEnhance.Contrast, 0.8, 1.2),
                              (ImageEnhance.Color, 0.85, 1.15)]
                random.shuffle(operations)
                for enhancer, low, high in operations:
                    image = enhancer(image).enhance(random.uniform(low, high))
            if random.random() < 0.15:
                image = image.filter(ImageFilter.GaussianBlur(random.uniform(0.2, 1.0)))
            if random.random() < 0.2:
                buffer = io.BytesIO()
                image.save(buffer, format='JPEG', quality=random.randint(55, 95))
                buffer.seek(0)
                with Image.open(buffer) as decoded:
                    image = decoded.convert('RGB').copy()
        array = np.array(image, dtype=np.float32, copy=True) / 255.
        tensor = torch.from_numpy(array).permute(2, 0, 1)
        tensor = (tensor - tensor.new_tensor(MEAN)[:, None, None]) / tensor.new_tensor(STD)[:, None, None]
        if self.augment and random.random() < 0.15:
            h, w = tensor.shape[-2:]
            area = random.uniform(0.01, 0.05) * h * w
            ratio = random.uniform(0.5, 2.0)
            rh, rw = int(math.sqrt(area / ratio)), int(math.sqrt(area * ratio))
            if 0 < rh <= h and 0 < rw <= w:
                y, x = random.randint(0, h-rh), random.randint(0, w-rw)
                tensor[:, y:y+rh, x:x+rw] = 0  # Normalized channel mean.
        return tensor.contiguous()


class ImageList(Dataset):
    def __init__(self, paths, size, augment=False, boxes=None):
        self.paths, self.size, self.boxes = paths, size, boxes
        self.transform = ImageTransform(augment)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = str(self.paths[index])
        cached = IMAGE_CACHE.get(path)
        if (isinstance(cached,np.ndarray) and self.boxes is None
                and IMAGE_CACHE_INFO.get('image_size') == self.size):
            image = Image.fromarray(cached).copy()
            return self.transform(image).unsqueeze(0)
        source_path = io.BytesIO(cached) if isinstance(cached,bytes) else path
        with Image.open(source_path) as source:
            image = source.convert('RGB')
        original_long = max(image.size)
        target = self.size
        if self.boxes is not None:
            image = image.crop(self.boxes[index])
            # Preserve the upstream query-crop resize protocol.
            target = max(1, int(self.size * max(image.size) / original_long))
        image.thumbnail((target, target), Image.Resampling.LANCZOS)
        return self.transform(image).unsqueeze(0)


def identity_collate(items):
    return items




### 2.5. Trích descriptor, loss và hard-negative mining

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
def forward_images(model, images, microbatch, pixel_limit):
    groups, vectors = defaultdict(list), [None] * len(images)
    device = next(model.parameters()).device
    for i, image in enumerate(images):
        groups[tuple(image.shape[1:])].append(i)
    for shape, indices in groups.items():
        limit = max(1, min(microbatch, pixel_limit // (shape[-2] * shape[-1])))
        for start in range(0, len(indices), limit):
            selection = indices[start:start+limit]
            batch = torch.cat([images[i].to(device, non_blocking=True) for i in selection])
            output = model(batch)
            for j, index in enumerate(selection):
                vectors[index] = output[:, j:j+1]
    return torch.cat(vectors, dim=1)


@torch.no_grad()
def extract(model, paths, cfg, boxes=None):
    model.eval()
    data = ImageList(paths, cfg['image_size'], boxes=boxes)
    loader = DataLoader(data, batch_size=cfg['extraction_window'], shuffle=False,
                        **loader_options(cfg), pin_memory=next(model.parameters()).is_cuda,
                        collate_fn=identity_collate, worker_init_fn=seed_worker,
                        generator=torch.Generator().manual_seed(0))
    device = next(model.parameters()).device
    result = torch.empty((model.output_dim, len(paths)), device=device)
    offset = 0
    for images in loader:
        vectors = forward_images(model, images, cfg['microbatch'], cfg['max_batch_pixels'])
        result[:, offset:offset+len(images)] = vectors
        offset += len(images)
    return result


def tuple_loss(vectors, margin=0.7):
    tuples = vectors.reshape(vectors.shape[0], -1, 7)
    distances = (tuples[:, :, :1] - tuples[:, :, 1:] + 1e-6).square().sum(0).sqrt()
    return 0.5 * (distances[:, 0].square().sum() + (margin-distances[:, 1:]).clamp(min=0).square().sum())


@torch.no_grad()
def mine(model, part, cfg, seed, validation=False, smoke=False):
    started = time.monotonic()
    generator = torch.Generator().manual_seed(seed)
    qsize = len(part['qidxs']) if validation else min(cfg['query_size'], len(part['qidxs']))
    if smoke:
        qsize = cfg['batch_size']
    sampled = torch.randperm(len(part['qidxs']), generator=generator)[:qsize].tolist()
    qs = [int(part['qidxs'][i]) for i in sampled]
    ps = [int(part['pidxs'][i]) for i in sampled]
    poolsize = len(part['paths']) if validation else min(cfg['pool_size'], len(part['paths']))
    if smoke:
        poolsize = min(256, len(part['paths']))
    pool = torch.randperm(len(part['paths']), generator=generator)[:poolsize].tolist()
    unique = list(dict.fromkeys(pool + qs + ps))
    positions = {index: pos for pos, index in enumerate(unique)}
    descriptors = extract(model, [part['paths'][i] for i in unique], cfg)
    qvectors = descriptors[:, [positions[i] for i in qs]]
    pvectors = descriptors[:, [positions[i] for i in pool]]
    ranks = torch.argsort(pvectors.T @ qvectors, dim=0, descending=True).cpu().numpy()
    tuples = []
    for column, (q, p) in enumerate(zip(qs, ps)):
        seen, negatives = {part['cluster'][q]}, []
        for rank in ranks[:, column]:
            candidate = pool[int(rank)]
            cluster = part['cluster'][candidate]
            if cluster not in seen:
                negatives.append(candidate)
                seen.add(cluster)
                if len(negatives) == 5:
                    break
        if len(negatives) != 5:
            raise RuntimeError('Mining pool has fewer than five distinct negative clusters.')
        tuples.append([q, p] + negatives)
    seconds = time.monotonic() - started
    print(f'Mining {"val" if validation else "train"}: {len(unique)} unique images, '
          f'{len(tuples)} tuples, {seconds/60:.1f} min', flush=True)
    if validation:
        return tuples, descriptors, positions, seconds
    return tuples, None, None, seconds




### 2.6. Training, validation và optimizer

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
class TupleImages(Dataset):
    def __init__(self, part, tuples, cfg):
        self.images = ImageList(part['paths'], cfg['image_size'], augment=True)
        self.tuples = tuples

    def __len__(self):
        return len(self.tuples)

    def __getitem__(self, index):
        return [self.images[i] for i in self.tuples[index]]


def train_epoch(model, optimizer, part, cfg, epoch, smoke=False):
    tuples, _, _, mining_seconds = mine(model, part, cfg, cfg['seed']+epoch, smoke=smoke)
    data = TupleImages(part, tuples, cfg)
    loader = DataLoader(data, batch_size=cfg['batch_size'], shuffle=True, drop_last=True,
                        **loader_options(cfg), pin_memory=next(model.parameters()).is_cuda,
                        collate_fn=identity_collate, worker_init_fn=seed_worker,
                        generator=torch.Generator().manual_seed(cfg['seed']+epoch))
    model.train()
    total, count, pending = 0., 0, []
    for step, batch in enumerate(loader, 1):
        pending.extend(optimizer_batch(model, optimizer, batch, cfg))
        count += len(batch)
        if step == 1 or step % 20 == 0 or step == len(loader):
            total += torch.stack(pending).double().sum().item()
            pending.clear()
            print(f'Epoch {epoch+1} train {step}/{len(loader)}: loss/tuple={total/count:.6f}', flush=True)
    if not count:
        raise RuntimeError('No full training batch')
    return total/count, mining_seconds, len(loader)


def optimizer_batch(model, optimizer, batch, cfg):
    """Shared measured training step: same code in training and GPU profiler."""
    optimizer.zero_grad(set_to_none=True)
    losses,_ = backward_tuples(model,batch,cfg)
    grads = [p.grad for p in model.parameters() if p.grad is not None]
    if not grads or not torch.stack([g.isfinite().all() for g in grads]).all().item():
        raise FloatingPointError('Non-finite or missing gradients; aborting optimizer update.')
    optimizer.step()
    if not torch.isfinite(model.p).all().item() or model.p.item() <= 0:
        raise FloatingPointError('Invalid learned GeM p')
    return losses


def backward_tuples(model,batch,cfg,collect_descriptors=False):
    fusion = cfg.get('tuple_fusion',1)
    if fusion not in {1,2,5}:
        raise ValueError('tuple_fusion must be 1, 2 or 5; effective batch remains five tuples')
    if fusion > 1 and any(isinstance(m,(nn.BatchNorm1d,nn.BatchNorm2d,nn.BatchNorm3d,
                                       nn.Dropout,nn.Dropout2d,nn.Dropout3d)) for m in model.modules()):
        raise ValueError('Tuple fusion requires this VGG/attention model without batchnorm/dropout')
    losses,descriptors = [],[]
    for start in range(0,len(batch),fusion):
        images = [image for item in batch[start:start+fusion] for image in item]
        vectors = forward_images(model, images, cfg['microbatch'], cfg['max_batch_pixels'])
        loss = tuple_loss(vectors, cfg['margin'])
        loss.backward()  # Summed tuple objective; optimizer still steps once after five tuples.
        losses.append(loss.detach())
        if collect_descriptors:
            descriptors.append(vectors.detach().float().cpu())
    return losses,descriptors


@torch.no_grad()
def validate(model, part, cfg, smoke=False):
    tuples, vectors, positions, mining_seconds = mine(
        model, part, cfg, cfg['seed']+100000, validation=True, smoke=smoke)
    # Fixed pair sampling seed; negatives still change with the current model.
    stop = len(tuples) // cfg['batch_size'] * cfg['batch_size']
    if not stop:
        raise RuntimeError('No full validation batch')
    total = 0.
    for start in range(0, stop, cfg['batch_size']):
        indices = [positions[i] for t in tuples[start:start+cfg['batch_size']] for i in t]
        total += tuple_loss(vectors[:, indices], cfg['margin']).item()
    return total/stop, mining_seconds


def make_optimizer(model, cfg):
    groups = [dict(params=model.features.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay']),
              dict(params=[model.p], lr=cfg['lr']*10, weight_decay=0)]
    if cfg['attention']:
        groups.append(dict(params=model.attention.parameters(), lr=cfg['attention_lr'], weight_decay=0))
    return torch.optim.Adam(groups)


def set_epoch_lr(optimizer, cfg, epoch):
    # Explicit historical pre-epoch exponential schedule, without scheduler.step warning.
    base = [cfg['lr'], cfg['lr']*10] + ([cfg['attention_lr']] if cfg['attention'] else [])
    for group, lr in zip(optimizer.param_groups, base):
        group['lr'] = lr * math.exp(-0.01 * (epoch+1))




### 2.7. Lưu checkpoint và resume

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
def cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return value


def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])


def restore_rng(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if state['cuda'] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([item.cpu() for item in state['cuda']])


def save_checkpoint(path, state, scratch):
    """Serialize locally, copy+hash+rename on Drive. Keep previous latest for recovery."""
    path, scratch = Path(path), Path(scratch)
    path.parent.mkdir(parents=True, exist_ok=True)
    scratch.mkdir(parents=True, exist_ok=True)
    local = scratch / f'{path.stem}_{time.time_ns()}.pt'
    pending = path.with_name(path.name + '.pending')
    try:
        torch.save(cpu_tree(state), local)
        digest = sha256(local)
        shutil.copyfile(local, pending)
        if sha256(pending) != digest:
            raise IOError(f'Drive checkpoint copy failed hash verification: {pending}')
        if path.name == 'latest.pt' and path.exists():
            path.replace(path.with_name('previous_latest.pt'))
            old_digest = path.with_name('latest.pt.sha256')
            if old_digest.exists():
                old_digest.replace(path.with_name('previous_latest.pt.sha256'))
        pending.replace(path)
        path.with_name(path.name + '.sha256').write_text(digest + '\n')
    finally:
        local.unlink(missing_ok=True)


def publish_checkpoint_alias(source, destination):
    source, destination = Path(source), Path(destination)
    pending = destination.with_name(destination.name + '.pending')
    digest = sha256(source)
    shutil.copyfile(source, pending)
    if sha256(pending) != digest:
        raise IOError('Drive alias copy failed verification')
    pending.replace(destination)
    destination.with_name(destination.name + '.sha256').write_text(digest + '\n')


def load_checkpoint(path, identity):
    path = Path(path)
    digest_file = path.with_name(path.name + '.sha256')
    if not digest_file.exists() or sha256(path) != digest_file.read_text().strip():
        raise IOError(f'Missing/mismatched checkpoint SHA256: {path}. Try previous_latest.pt explicitly.')
    state = torch.load(path, map_location='cpu', weights_only=False)  # Trusted own checkpoint only.
    if state.get('format') != VERSION or state.get('identity') != identity:
        raise ValueError('Checkpoint run/source/data/config mismatch; never resume the other run.')
    return state




### 2.8. Benchmark và xuất descriptor

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
def discover_benchmarks(roots):
    result = {}
    for name in ['oxford5k', 'paris6k']:
        matches = [p for root in roots for p in Path(root).rglob(f'gnd_{name}.pkl')]
        if not matches:
            continue
        if len(matches) != 1:
            raise ValueError(f'Ambiguous {name} metadata: {matches}')
        path = matches[0]
        with path.open('rb') as f:
            gt = pickle.load(f)
        images = path.parent / 'jpg'
        names = set(gt['imlist']) | set(gt['qimlist'])
        for item in names:
            assert isinstance(item, str) and Path(item).name == item
            if not (images / (item+'.jpg')).is_file():
                raise FileNotFoundError(images / (item+'.jpg'))
        result[name] = dict(gt=gt, root=images, metadata_sha256=sha256(path))
    return result


def benchmark(model, benchmarks, cfg, output, compute_map, export=False):
    result = {}
    for name, info in benchmarks.items():
        print('Benchmark:', name, flush=True)
        gt, root = info['gt'], info['root']
        gallery = extract(model, [str(root/(n+'.jpg')) for n in gt['imlist']], cfg).cpu()
        query = extract(model, [str(root/(n+'.jpg')) for n in gt['qimlist']], cfg,
                        boxes=[tuple(g['bbx']) for g in gt['gnd']]).cpu()
        ranks = np.argsort(-(gallery.numpy().T @ query.numpy()), axis=0)
        score, aps, _, _ = compute_map(ranks, gt['gnd'])
        result[name] = dict(map=float(score), aps=np.asarray(aps).tolist(),
                            protocol='legacy_ok_junk', scale=[1], whitening=False, query_expansion=False,
                            metadata_sha256=info['metadata_sha256'])
        if export:
            # Small descriptor files are saved on Drive for later CPU search experiments.
            np.savez_compressed(Path(output).parent / f'{name}_best_descriptors.npz',
                                gallery=gallery.numpy(), query=query.numpy(),
                                gallery_ids=np.asarray(gt['imlist']), query_ids=np.asarray(gt['qimlist']))
        print(name, f'mAP={score*100:.3f}%', flush=True)
    atomic_json(output, result)
    return result




### 2.9. Smoke test

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
def _run_smoke_impl(cfg, partitions, weights, device, run_dir, identity, scratch):
    seed_all(cfg['seed'])
    model = make_model(cfg, weights, device)
    optimizer = make_optimizer(model, cfg)
    before = model.features[0].weight.detach().clone()
    started = time.monotonic()
    train_loss, _, _ = train_epoch(model, optimizer, partitions['train'], cfg, 0, smoke=True)
    val_loss, _ = validate(model, partitions['val'], cfg, smoke=True)
    assert math.isfinite(train_loss) and math.isfinite(val_loss)
    assert not torch.equal(before, model.features[0].weight)
    if cfg['attention']:
        assert torch.count_nonzero(model.attention.conv.weight).item() > 0
    path = Path(run_dir) / 'smoke' / 'roundtrip.pt'
    state = dict(format=VERSION, identity=identity, model=model.state_dict(), optimizer=optimizer.state_dict())
    save_checkpoint(path, state, scratch)
    restored = load_checkpoint(path, identity)
    assert all(torch.equal(t.cpu(), restored['model'][key]) for key, t in model.state_dict().items())
    optimizer.load_state_dict(restored['optimizer'])
    result = dict(status='PASS', train_loss=train_loss, val_loss=val_loss,
                  seconds=time.monotonic()-started, image_size=cfg['image_size'])
    atomic_json(Path(run_dir)/'smoke'/'result.json', result)
    del model, optimizer, state, restored, before
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    print('SMOKE PASS:', result, flush=True)
    return result


def run_smoke(cfg, partitions, weights, device, run_dir, identity, scratch):
    try:
        return _run_smoke_impl(cfg, partitions, weights, device, run_dir, identity, scratch)
    except Exception as error:
        traceback.print_exc()
        failure = dict(status='FAILED', error_type=type(error).__name__, error=str(error),
                       image_size=cfg['image_size'], note='Full training was not started.')
        atomic_json(Path(run_dir)/'smoke'/'result.json', failure)
        # Clear exception frames holding CUDA activations before a notebook retains the error.
        traceback.clear_frames(error.__traceback__)
        error.__traceback__ = None
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        raise RuntimeError('Smoke failed; inspect Drive logs and smoke/result.json. No full training started.') from None




### 2.10. Controller chạy epoch và ngân sách

Code Python dưới đây ghi vào module local; chưa khởi chạy training.


In [ ]:
%%writefile -a /content/iir_colab_code/colab_runtime.py
def train_run(cfg, partitions, weights, run_dir, identity, operational, benchmarks, compute_map, *, device=None):
    # CPU override is used only by the local integration verifier, not notebook cells.
    device = torch.device(device) if device is not None else torch.device('cuda:0')
    if device.type == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('Choose a GPU runtime in Colab before training.')
    run_dir = Path(run_dir)
    scratch = Path(operational['scratch'])
    started = time.monotonic()
    cache_info = prepare_image_cache(partitions['train']['paths']+partitions['val']['paths'],cfg)
    atomic_json(run_dir/'image_cache.json',cache_info)
    latest = run_dir/'latest.pt'
    resume = operational['resume']
    if resume == 'auto':
        resume = latest if latest.exists() else None
        if resume is None and any((run_dir/name).exists() for name in ['previous_latest.pt', 'history.json']):
            raise RuntimeError('Existing run has no latest.pt. Select previous_latest.pt explicitly or a new RUN_ID.')
    elif resume:
        resume = Path(resume)
        if resume.resolve().parent != run_dir.resolve():
            raise ValueError('Resume from this RUN_ID folder; latest/best must remain together.')
    if resume is None and any((run_dir/name).exists() for name in ['latest.pt', 'previous_latest.pt', 'history.json']):
        raise RuntimeError('Existing latest.pt; use auto resume or a new RUN_ID.')
    state = load_checkpoint(resume, identity) if resume else None
    # Versioned best files allow an interrupted latest/best publication to recover.
    best_file = state['best_file'] if state else None
    best_path = run_dir/best_file if best_file else run_dir/'best.pt'
    if state and not best_path.exists():
        raise FileNotFoundError('Keep the versioned best checkpoint beside latest.pt on Drive.')
    start_epoch = state['epoch'] if state else 0
    if cfg['epochs'] < start_epoch:
        raise ValueError('Epoch target is below the checkpoint epoch.')
    previous_seconds = state.get('total_session_seconds', 0) if state else 0
    usage_file = run_dir/'usage_total.json'
    if usage_file.exists():
        previous_seconds = max(previous_seconds, json.loads(usage_file.read_text())['total_session_seconds'])
    rate = operational['cu_per_hour']
    cu_cap = operational['run_cu_budget']
    wall_cap = operational['session_hours'] * 3600

    def remaining():
        elapsed = time.monotonic()-started
        seconds = wall_cap-elapsed
        if rate is not None:
            seconds = min(seconds, cu_cap/rate*3600 - previous_seconds-elapsed)
        return seconds

    if remaining() < 600:
        raise RuntimeError('Less than ten minutes of configured session/CU budget remains.')
    # Complete runs can be reopened for evaluation without paying for another smoke.
    if start_epoch < cfg['epochs']:
        try:
            run_smoke(cfg, partitions, weights, device, run_dir, identity, scratch)
        finally:
            atomic_json(usage_file, dict(total_session_seconds=previous_seconds+time.monotonic()-started))
    seed_all(cfg['seed'])
    model = make_model(cfg, weights, device)
    optimizer = make_optimizer(model, cfg)
    history, best_loss = [], float('inf')
    if state:
        model.load_state_dict(state['model'], strict=True)
        optimizer.load_state_dict(state['optimizer'])
        history, best_loss = state['history'], state['best_loss']
        restore_rng(state['rng'])
        best_state = load_checkpoint(best_path, identity)
        assert best_state['best_loss'] == best_loss
        del best_state, state
        publish_checkpoint_alias(best_path, run_dir/'best.pt')
    atomic_json(run_dir/'run_record.json', dict(identity=identity, config=cfg, operational=operational,
                environment=dict(python=sys.version, torch=torch.__version__, numpy=np.__version__,
                                 gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU_VERIFIER', precision=cfg.get('precision','FP32')),
                resume=str(resume), selection='clean_validation_contrastive_loss',
                validation_pairs_processed=180, gradient_reduction='sum per tuple; five tuples per update',
                schedule='lr_group * exp(-0.01 * (epoch_index+1)); historical pre-epoch convention'))
    status = 'COMPLETED'
    try:
        for epoch in range(start_epoch, cfg['epochs']):
            estimate = history[-1].get('total_seconds', history[-1]['train_val_seconds']*1.25) if history else 0
            if estimate and remaining() < 1.2*estimate + 300:
                status = 'PAUSED_BEFORE_EPOCH_FOR_BUDGET'
                print(status, 'Resume this same notebook/RUN_ID in a new session.', flush=True)
                break
            epoch_started = time.monotonic()
            seed_all(cfg['seed']+epoch)
            if device.type == 'cuda':
                torch.cuda.reset_peak_memory_stats()
            set_epoch_lr(optimizer, cfg, epoch)
            print(f'\n=== Epoch {epoch+1}/{cfg["epochs"]}; LR {[g["lr"] for g in optimizer.param_groups]} ===', flush=True)
            train_start = time.monotonic()
            train_loss, train_mining, steps = train_epoch(model, optimizer, partitions['train'], cfg, epoch)
            train_seconds = time.monotonic()-train_start
            val_start = time.monotonic()
            val_loss, val_mining = validate(model, partitions['val'], cfg)
            if not math.isfinite(train_loss) or not math.isfinite(val_loss):
                raise FloatingPointError('Non-finite epoch loss')
            improved = val_loss < best_loss
            best_loss = min(best_loss, val_loss)
            if improved:
                best_file = f'best_epoch_{epoch+1:03d}.pt'
                best_path = run_dir/best_file
            row = dict(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss, gem_p=model.p.item(),
                       optimizer_steps=steps, train_seconds=train_seconds, train_mining_seconds=train_mining,
                       val_seconds=time.monotonic()-val_start, val_mining_seconds=val_mining,
                       peak_gpu_gib=torch.cuda.max_memory_allocated()/1024**3 if device.type == 'cuda' else None,
                       train_val_seconds=time.monotonic()-epoch_started)
            history.append(row)
            checkpoint = dict(format=VERSION, identity=identity, epoch=epoch+1,
                              model=model.state_dict(), optimizer=optimizer.state_dict(),
                              schedule=dict(gamma=math.exp(-0.01), timing='pre_epoch', completed_epochs=epoch+1),
                              rng=rng_state(), best_loss=best_loss, best_file=best_file, history=history,
                              total_session_seconds=previous_seconds+time.monotonic()-started)
            # Publish best before latest, so a failed best upload cannot silently continue.
            if improved:
                save_checkpoint(best_path, checkpoint, scratch)
            save_checkpoint(latest, checkpoint, scratch)
            if improved:
                publish_checkpoint_alias(best_path, run_dir/'best.pt')
            del checkpoint
            atomic_json(run_dir/'history.json', history)
            if epoch+1 == 7:
                shutil.copyfile(latest, run_dir/'epoch_007.pt')
                shutil.copyfile(run_dir/'latest.pt.sha256', run_dir/'epoch_007.pt.sha256')
            benchmark_file = run_dir/f'evaluation_epoch_{epoch+1:03d}.json'
            if benchmarks and (epoch+1) % operational['eval_every'] == 0 and remaining() > 1800:
                benchmark(model, benchmarks, cfg, benchmark_file, compute_map)
            row['total_seconds'] = time.monotonic()-epoch_started
            atomic_json(run_dir/'history.json', history)
            atomic_json(run_dir/'timing_latest.json', dict(last_epoch=row,
                        total_session_seconds=previous_seconds+time.monotonic()-started,
                        estimated_cu_used=(previous_seconds+time.monotonic()-started)*rate/3600 if rate else None))
            print('Epoch complete:', json.dumps(row), flush=True)
        if operational['final_evaluation'] and benchmarks and best_path.exists():
            if remaining() > 1800:
                best = load_checkpoint(best_path, identity)
                model.load_state_dict(best['model'], strict=True)
                result = benchmark(model, benchmarks, cfg, run_dir/'evaluation_best.json', compute_map, export=True)
                atomic_json(run_dir/'evaluation_best_identity.json', dict(epoch=best['epoch'],
                            checkpoint_sha256=sha256(best_path), metrics=result, identity=identity))
                del best
            else:
                print('Best-checkpoint evaluation deferred: low time/CU budget.', flush=True)
        elif not benchmarks:
            print('Oxford/Paris inputs absent: training/validation complete; no mAP claimed.', flush=True)
        result = dict(status=status, completed_epochs=len(history), target_epochs=cfg['epochs'],
                      best_val_loss=best_loss, latest=str(latest), best=str(run_dir/'best.pt'),
                      total_session_seconds=previous_seconds+time.monotonic()-started)
        atomic_json(run_dir/'training_result.json', result)
        print(result, flush=True)
        return result
    except BaseException as error:
        traceback.print_exc()
        atomic_json(run_dir/'interruption.json', dict(error_type=type(error).__name__, error=str(error),
                    recovery='Resume latest.pt (completed epoch). Partial current epoch is discarded.',
                    session_seconds=time.monotonic()-started))
        raise
    finally:
        atomic_json(usage_file, dict(total_session_seconds=previous_seconds+time.monotonic()-started,
                    note='Measured notebook execution wall time; CU is an estimate, not Colab billing.'))
        del model, optimizer
        gc.collect()
        torch.cuda.empty_cache()


def make_identity(cfg, data_identity, runtime_hash):
    # Epoch target can grow; all model/data/augmentation/learning choices must stay fixed.
    fixed = {key: value for key, value in cfg.items() if key != 'epochs'}
    return dict(version=VERSION, config=fixed, data=data_identity, runtime_sha256=runtime_hash,
                weights_sha256=WEIGHTS_HASH)


### 2.11. Evaluator mAP

Protocol legacy Oxford/Paris; dùng `np.isin` để tương thích NumPy mới.


In [ ]:
%%writefile /content/iir_colab_code/legacy_evaluate.py
import numpy as np

def compute_ap(ranks, nres):
    """
    Computes average precision for given ranked indexes.
    
    Arguments
    ---------
    ranks : zerro-based ranks of positive images
    nres  : number of positive images
    
    Returns
    -------
    ap    : average precision
    """

    # number of images ranked by the system
    nimgranks = len(ranks)

    # accumulate trapezoids in PR-plot
    ap = 0

    recall_step = 1. / nres

    for j in np.arange(nimgranks):
        rank = ranks[j]

        if rank == 0:
            precision_0 = 1.
        else:
            precision_0 = float(j) / rank

        precision_1 = float(j + 1) / (rank + 1)

        ap += (precision_0 + precision_1) * recall_step / 2.

    return ap

def compute_map(ranks, gnd, kappas=[]):
    """
    Computes the mAP for a given set of returned results.

         Usage: 
           map = compute_map (ranks, gnd) 
                 computes mean average precsion (map) only
        
           map, aps, pr, prs = compute_map (ranks, gnd, kappas) 
                 computes mean average precision (map), average precision (aps) for each query
                 computes mean precision at kappas (pr), precision at kappas (prs) for each query
        
         Notes:
         1) ranks starts from 0, ranks.shape = db_size X #queries
         2) The junk results (e.g., the query itself) should be declared in the gnd stuct array
         3) If there are no positive images for some query, that query is excluded from the evaluation
    """

    map = 0.
    nq = len(gnd) # number of queries
    aps = np.zeros(nq)
    pr = np.zeros(len(kappas))
    prs = np.zeros((nq, len(kappas)))
    nempty = 0

    for i in np.arange(nq):
        qgnd = np.array(gnd[i]['ok'])

        # no positive images, skip from the average
        if qgnd.shape[0] == 0:
            aps[i] = float('nan')
            prs[i, :] = float('nan')
            nempty += 1
            continue

        try:
            qgndj = np.array(gnd[i]['junk'])
        except:
            qgndj = np.empty(0)

        # sorted positions of positive and junk images (0 based)
        pos  = np.arange(ranks.shape[0])[np.isin(ranks[:,i], qgnd)]
        junk = np.arange(ranks.shape[0])[np.isin(ranks[:,i], qgndj)]

        k = 0;
        ij = 0;
        if len(junk):
            # decrease positions of positives based on the number of
            # junk images appearing before them
            ip = 0
            while (ip < len(pos)):
                while (ij < len(junk) and pos[ip] > junk[ij]):
                    k += 1
                    ij += 1
                pos[ip] = pos[ip] - k
                ip += 1

        # compute ap
        ap = compute_ap(pos, len(qgnd))
        map = map + ap
        aps[i] = ap

        # compute precision @ k
        pos += 1 # get it to 1-based
        for j in np.arange(len(kappas)):
            kq = min(max(pos), kappas[j]); 
            prs[i, j] = (pos <= kq).sum() / kq
        pr = pr + prs[i, :]

    map = map / (nq - nempty)
    pr = pr / (nq - nempty)

    return map, aps, pr, prs


def compute_map_and_print(dataset, ranks, gnd, kappas=[1, 5, 10]):
    
    # old evaluation protocol
    if dataset.startswith('oxford5k') or dataset.startswith('paris6k'):
        map, aps, _, _ = compute_map(ranks, gnd)
        print('>> {}: mAP {:.2f}'.format(dataset, np.around(map*100, decimals=2)))

    # new evaluation protocol
    elif dataset.startswith('roxford5k') or dataset.startswith('rparis6k'):
        
        gnd_t = []
        for i in range(len(gnd)):
            g = {}
            g['ok'] = np.concatenate([gnd[i]['easy']])
            g['junk'] = np.concatenate([gnd[i]['junk'], gnd[i]['hard']])
            gnd_t.append(g)
        mapE, apsE, mprE, prsE = compute_map(ranks, gnd_t, kappas)

        gnd_t = []
        for i in range(len(gnd)):
            g = {}
            g['ok'] = np.concatenate([gnd[i]['easy'], gnd[i]['hard']])
            g['junk'] = np.concatenate([gnd[i]['junk']])
            gnd_t.append(g)
        mapM, apsM, mprM, prsM = compute_map(ranks, gnd_t, kappas)

        gnd_t = []
        for i in range(len(gnd)):
            g = {}
            g['ok'] = np.concatenate([gnd[i]['hard']])
            g['junk'] = np.concatenate([gnd[i]['junk'], gnd[i]['easy']])
            gnd_t.append(g)
        mapH, apsH, mprH, prsH = compute_map(ranks, gnd_t, kappas)

        print('>> {}: mAP E: {}, M: {}, H: {}'.format(dataset, np.around(mapE*100, decimals=2), np.around(mapM*100, decimals=2), np.around(mapH*100, decimals=2)))
        print('>> {}: mP@k{} E: {}, M: {}, H: {}'.format(dataset, kappas, np.around(mprE*100, decimals=2), np.around(mprM*100, decimals=2), np.around(mprH*100, decimals=2)))


### 2.12. Import runtime và mở log

Nạp các module vừa ghi. Training chỉ bắt đầu ở mục 4.


In [ ]:
RUNTIME_SOURCE = (LOCAL_CODE/'colab_runtime.py').read_text()
# IPython/Colab may append a terminal newline to a %%writefile cell.
# Canonicalize only terminal newlines; edits inside the evaluator still fail the hash.
EVALUATE_SOURCE = (LOCAL_CODE/'legacy_evaluate.py').read_text().rstrip(chr(10))
RUNTIME_HASH = hashlib.sha256(RUNTIME_SOURCE.encode()).hexdigest()
EVALUATE_HASH = hashlib.sha256(EVALUATE_SOURCE.encode()).hexdigest()
# Exact known transport variant: Colab/editor removed trailing whitespace on each line.
# Preserve the original evaluator identity so existing checkpoints can resume.
if EVALUATE_HASH == 'b2619397085e7dfcbdc78ff31abbc02a2c5ab76014c93eef84f20f3ce251dc6d':
    EVALUATE_HASH = 'ced1d110dd19f3f887e75f18ac9120aaff8900f5d6b52dc8c45c66c823141fd0'
assert RUNTIME_HASH == '9724c31d9f48d5be27192aa5761006e28b3977caa71c76b69ec899d3ae821ccc', "Code đã sửa hoặc append lặp: chạy lại toàn bộ nhóm cell code từ đầu; nếu chủ động sửa, cập nhật hash và dùng RUN_ID mới."
assert EVALUATE_HASH == 'ced1d110dd19f3f887e75f18ac9120aaff8900f5d6b52dc8c45c66c823141fd0', "Evaluator đã đổi; kiểm tra lại code (đã bỏ qua newline cuối file)."
if str(LOCAL_CODE) not in sys.path:
    sys.path.insert(0, str(LOCAL_CODE))
import importlib
import colab_runtime as rt
import legacy_evaluate
importlib.reload(rt)
importlib.reload(legacy_evaluate)
SESSION_LOG = rt.start_logging(RUN_DIR)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
print('Drive output:', RUN_DIR)
if CU_PER_HOUR is None:
    print('CU_PER_HOUR chưa nhập: chưa thể ước lượng ngân sách CU; chỉ áp dụng giới hạn giờ/session.')


## 3. Giải nén ZIP vào local và kiểm tra dataset

Copy ZIP Drive → local, kiểm tra CRC khi giải nén, từ chối path traversal/symlink và chỉ tái sử dụng extraction đã hoàn tất.
Kiểm tra đủ mọi đường dẫn SfM, counts, cặp positive, train/val không trùng cluster/CID; decode mẫu 128 ảnh.
Nếu có prepared manifest: kiểm tra metadata hashes và 64 image hashes. Ảnh ngoài mẫu vẫn có thể lỗi khi loader đọc sau đó.
Benchmark tùy chọn cần `oxford5k/gnd_oxford5k.pkl` + `oxford5k/jpg/`, tương tự `paris6k`.


In [ ]:
CACHE_ROOT = Path('/content/iir_colab_data')
SFM_ROOT = rt.extract_zip(SFM_ZIP, CACHE_ROOT)
BENCH_ROOTS = [rt.extract_zip(path, CACHE_ROOT) for path in BENCHMARK_ZIPS]
PARTITIONS, DATA_IDENTITY = rt.load_sfm(SFM_ROOT, SFM_METADATA_OVERRIDE, SFM_IMAGES_OVERRIDE)
WEIGHTS = rt.resolve_weights(WEIGHTS_PATH, Path(OUTPUT_ROOT)/'_weights_cache')
BENCHMARKS = rt.discover_benchmarks([SFM_ROOT] + BENCH_ROOTS)
print('Benchmarks:', list(BENCHMARKS) or 'None — chỉ train/validation; mAP đánh giá sau')

CFG = dict(attention=ATTENTION, attention_lr=ATTENTION_LR, epochs=EPOCHS, seed=SEED,
           image_size=IMAGE_SIZE, query_size=QUERY_SIZE, pool_size=POOL_SIZE,
           batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, margin=MARGIN,
           workers=WORKERS, microbatch=MICROBATCH, max_batch_pixels=MAX_BATCH_PIXELS,
           extraction_window=EXTRACTION_WINDOW,prefetch_factor=PREFETCH_FACTOR,cache_mode=CACHE_MODE,
           cache_gib=CACHE_GIB,tuple_fusion=TUPLE_FUSION,
           augmentation_version='mild_photo_blur_jpeg_erase_v1', precision=PRECISION)
DATA_IDENTITY['benchmark_metadata'] = {name: value['metadata_sha256'] for name, value in BENCHMARKS.items()}
IDENTITY = rt.make_identity(CFG, DATA_IDENTITY, RUNTIME_HASH)
IDENTITY['run_id'] = RUN_ID
IDENTITY['evaluator_sha256'] = EVALUATE_HASH
# Benchmark inputs are evaluation-only; adding them later must not invalidate model resume.
IDENTITY['data'].pop('benchmark_metadata', None)
OPERATIONAL = dict(resume=RESUME, scratch='/content/iir_checkpoint_staging',
                   cu_per_hour=CU_PER_HOUR, run_cu_budget=RUN_CU_BUDGET,
                   session_hours=SESSION_HOURS, eval_every=EVAL_EVERY,
                   final_evaluation=FINAL_EVALUATION)

# Refuse to overwrite provenance for an existing run before validating its identity.
identity_path = RUN_DIR/'experiment_identity.json'
if identity_path.exists() and json.loads(identity_path.read_text()) != IDENTITY:
    raise RuntimeError('RUN_ID đã có config/source/data khác. Đặt RUN_ID mới; không ghi đè run cũ.')
rt.atomic_json(identity_path, IDENTITY)
(RUN_DIR/'colab_runtime.py').write_text(RUNTIME_SOURCE)
(RUN_DIR/'legacy_evaluate.py').write_text(EVALUATE_SOURCE)
rt.atomic_json(RUN_DIR/'input_sources.json', dict(sfm_zip=SFM_ZIP, benchmark_zips=BENCHMARK_ZIPS,
               weights=str(WEIGHTS), benchmarks=list(BENCHMARKS)))
print(json.dumps(CFG, indent=2))


## 4. Smoke → training → checkpoint về Drive

- Smoke riêng ở resolution 1024: một batch có mining/backward/update, validation và checkpoint round-trip.
- Model full training được khởi tạo lại từ pretrained, không dùng trọng số smoke.
- Mining/validation dùng ảnh sạch. Augmentation chỉ dùng trong tuple training.
- Mỗi epoch: 2.000 tuple → 400 Adam updates → clean validation 180 tuple → lưu checkpoint → benchmark nếu đến lịch.
- GeM p dùng LR gấp 10 backbone. Run 2 có gate spatial-only 7×7, 99 tham số, LR riêng 1e-4; gate khởi tạo identity bằng `2*sigmoid`.
- Best chọn theo validation loss. Epoch 7 được giữ riêng để đối chiếu run cũ.
- Trước epoch tiếp theo, dùng timing epoch trước để dự tính còn đủ giờ/CU không. Đây là ước lượng, không phải đồng hồ billing Colab.
- Gián đoạn giữa epoch: giữ checkpoint epoch hoàn tất trước đó; chạy lại cùng RUN_ID với `RESUME='auto'`.
- Không copy checkpoint run 1 sang run 2. Khi kéo dài training chỉ tăng EPOCHS; thay config học/augmentation/source phải dùng RUN_ID mới.

Checkpoint được serialize ở local, copy sang Drive bằng file `.pending`, kiểm tra SHA256 rồi rename.
Drive mount không thay thế một hệ lưu trữ giao dịch; giữ `previous_latest.pt` và các best version để phục hồi nếu bị ngắt lúc ghi.


In [ ]:
if RUN_TRAINING:
    try:
        TRAINING_RESULT = rt.train_run(CFG, PARTITIONS, WEIGHTS, RUN_DIR, IDENTITY,
                                       OPERATIONAL, BENCHMARKS, legacy_evaluate.compute_map)
    except BaseException:
        import traceback
        traceback.print_exc()  # Also persist smoke/resume failures in the Drive session log.
        sys.stdout.flush()
        sys.stderr.flush()
        raise
else:
    print('RUN_TRAINING=False: chuẩn bị xong, chưa train.')


## 5. Kiểm tra output / đánh giá riêng sau training

`logs/session_*.log`: log trực tiếp trên Drive. `latest.pt`: resume; `previous_latest.pt`: dự phòng.
`best.pt` và `best_epoch_*.pt`: model chọn bởi validation loss; giữ cả versioned best khi resume.
`history.json`, `timing_latest.json`, `training_result.json`, `run_record.json`: loss/timing/config.
Nếu có Oxford/Paris, final evaluation xuất descriptor `.npz` về Drive để thử search bằng CPU sau.

Bạn có thể thêm benchmark ZIP sau này, đặt RUN_TRAINING=False và chạy cell đánh giá riêng bên dưới.
Cell này không cập nhật model và không chạy smoke. Việc trích descriptor vẫn nên dùng GPU.


In [ ]:
# Chỉ bật khi muốn đánh giá riêng best.pt; mặc định không chạy lặp sau training.
EVALUATE_BEST_ONLY = False
if EVALUATE_BEST_ONLY:
    if not BENCHMARKS:
        raise RuntimeError('Điền BENCHMARK_ZIPS và chạy lại bước dữ liệu trước.')
    checkpoint = rt.load_checkpoint(RUN_DIR/'best.pt', IDENTITY)
    eval_model = rt.make_model(CFG, WEIGHTS, torch.device('cuda:0'))
    try:
        eval_model.load_state_dict(checkpoint['model'], strict=True)
        metrics = rt.benchmark(eval_model, BENCHMARKS, CFG, RUN_DIR/'evaluation_best.json',
                               legacy_evaluate.compute_map, export=True)
        rt.atomic_json(RUN_DIR/'evaluation_best_identity.json', dict(epoch=checkpoint['epoch'],
                       checkpoint_sha256=rt.sha256(RUN_DIR/'best.pt'), metrics=metrics, identity=IDENTITY))
    finally:
        del eval_model, checkpoint
        import gc
        gc.collect()
        torch.cuda.empty_cache()

print('Output trên Drive:', RUN_DIR)
for name in ['latest.pt', 'best.pt', 'history.json', 'training_result.json', 'timing_latest.json']:
    path = RUN_DIR/name
    print(name, f'{path.stat().st_size/1024**2:.1f} MiB' if path.exists() else 'chưa có')
